# DealSight AI — Exploratory Data Analysis (EDA)
## Predictive Deal Intelligence & B2B Sales Dynamics

This notebook explores the historical sales interaction dataset to answer **10 core business questions** driving deal progression and deal loss.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('../data/sales_data.csv')
print(f"Loaded {df.shape[0]} deals across {df.shape[1]} features.")
df.head()

### 1. Deal Outcome Distribution
**Business Question:** *What is the baseline pipeline conversion rate?*

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='deal_outcome', palette=['#e11d48', '#10b981'], ax=ax)
ax.set_title('Deal Outcome Distribution (Progressed vs Lost)')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())} ({p.get_height()/len(df):.1%})',
                (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold')
plt.show()

### 2. Deal Value Distribution by Outcome
**Business Question:** *Are higher-value enterprise deals harder to progress?*

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=df, x='deal_outcome', y='deal_value', palette=['#f43f5e', '#10b981'], ax=ax)
ax.set_title('Deal Value Distribution Across Outcomes')
ax.set_ylabel('Deal Value ($ USD)')
plt.show()

### 3. Customer Sentiment vs Outcome
**Business Question:** *Does early customer sentiment predict deal success?*

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.kdeplot(data=df, x='sentiment_score', hue='deal_outcome', fill=True, common_norm=False, palette=['#ef4444', '#10b981'])
plt.title('Sentiment Density: Progressed vs Lost Deals')
plt.xlabel('Sentiment Score (0.0 to 1.0)')
plt.show()

### 4. Objections vs Outcome
**Business Question:** *Does repeated price objection correlate with deal loss?*

In [ ]:
obj_rates = df.groupby('price_objections')['deal_outcome'].value_counts(normalize=True).unstack().fillna(0)
obj_rates.plot(kind='bar', stacked=True, color=['#ef4444', '#10b981'], figsize=(9, 4))
plt.title('Progression Rate by Number of Price Objections')
plt.ylabel('Proportion')
plt.xlabel('Price Objections Raised')
plt.legend(title='Outcome')
plt.show()

### 5. Competitor Mentions vs Outcome
**Business Question:** *How much does active competitive evaluation lower win rate?*

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=df, x='competitor_mentions', y=(df['deal_outcome'] == 'Progressed').astype(int), palette='Blues_d', ax=ax)
ax.set_title('Win/Progression Rate vs Competitor Mentions')
ax.set_ylabel('Progress Rate')
ax.set_xlabel('Competitors Mentioned')
plt.show()

### 6. Decision Maker Presence vs Outcome
**Business Question:** *Does executive decision-maker attendance make or break the deal?*

In [ ]:
dm_impact = df.groupby('decision_maker_present')['deal_outcome'].value_counts(normalize=True).unstack()
dm_impact.plot(kind='bar', color=['#f43f5e', '#10b981'], figsize=(7, 4))
plt.title('Deal Progression Rate by Decision Maker Presence (0=No, 1=Yes)')
plt.ylabel('Ratio')
plt.xticks(ticks=[0, 1], labels=['DM Absent', 'DM Present'], rotation=0)
plt.show()

### 7. Demo Completion vs Outcome
**Business Question:** *Is successful product demonstration a mandatory milestone?*

In [ ]:
demo_impact = df.groupby('demo_completed')['deal_outcome'].value_counts(normalize=True).unstack()
demo_impact.plot(kind='bar', color=['#f43f5e', '#10b981'], figsize=(7, 4))
plt.title('Progression Rate by Product Demo Completion')
plt.xticks(ticks=[0, 1], labels=['Demo Pending', 'Demo Completed'], rotation=0)
plt.ylabel('Proportion')
plt.show()

### 8. Response Delay vs Outcome
**Business Question:** *Does prospect response lag signal declining deal momentum?*

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=df, x='deal_outcome', y='response_delay_hours', palette=['#ef4444', '#10b981'], ax=ax)
ax.set_title('Customer Response Delay (Hours) by Outcome')
ax.set_ylabel('Average Response Delay (Hours)')
plt.show()

### 9. Engagement Score vs Outcome
**Business Question:** *How strong is the correlation between engagement and progression?*

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(data=df, x='engagement_score', hue='deal_outcome', kde=True, bins=25, palette=['#ef4444', '#10b981'], ax=ax)
ax.set_title('Engagement Score Distribution Across Outcomes')
plt.show()

### 10. Deal Stage vs Outcome
**Business Question:** *At which sales pipeline stage do deals drop out the most?*

In [ ]:
stage_order = ['Discovery', 'Demo', 'Evaluation', 'Negotiation', 'Proposal']
stage_rates = df.groupby('deal_stage')['deal_outcome'].value_counts(normalize=True).unstack().reindex(stage_order)
stage_rates.plot(kind='bar', stacked=True, color=['#f43f5e', '#10b981'], figsize=(10, 4))
plt.title('Pipeline Conversion & Drop-off Rates by Sales Stage')
plt.ylabel('Proportion')
plt.xlabel('Deal Stage')
plt.show()

## Key Business Takeaways for Machine Learning Pipeline
1. **Decision Maker Presence** and **Demo Completion** are the two strongest positive categorical predictors.
2. **Price Objections > 2** drops deal progression probability below 20%.
3. **Response Delay > 48 hours** strongly correlates with stalled deals.
4. Domain-engineered composite features like `objection_pressure` and `decision_maker_score` capture interaction non-linearities cleanly.